# **Install Dependencies**

In [1]:
!pip install facenet-pytorch opencv-python-headless -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 802.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5

# **Import Modules**

In [2]:
import kagglehub
import cv2
import os
import random
import shutil

from tqdm import tqdm


# **Download Dataset**

In [3]:
# Download latest version
DATASET_PATH = kagglehub.dataset_download("axondata/face-anti-spoofing-dataset")

print("Path to dataset files:", DATASET_PATH)

100%|██████████| 4.94G/4.94G [00:59<00:00, 89.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/axondata/face-anti-spoofing-dataset/versions/1


# **Extract Dataset**

In [ ]:
VIDEO_EXTENSIONS = (".mp4", ".mov", ".MOV", ".MP4", ".avi", ".AVI")
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

DATASET_PATH = "./dataset"
OUTPUT_ROOT  = "./output"
FOLDER_LABEL = {
    "real": 0,  # live
    "fake": 1,  # spoof
}

In [4]:
class DatasetBuilder:

    def __init__(self, dataset_path, output_root, folder_label, frame_interval=10):
        self.dataset_path   = dataset_path
        self.output_root    = output_root
        self.folder_label   = folder_label
        self.frame_interval = frame_interval

    def build(self):
        for folder, label in self.folder_label.items():
            folder_path = os.path.join(self.dataset_path, folder)
            label_name  = "live" if label == 0 else "spoof"
            output_dir  = os.path.join(self.output_root, label_name)
            os.makedirs(output_dir, exist_ok=True)

            all_files = self._crawl(folder_path)

            videos = [f for f in all_files if f.endswith(VIDEO_EXTENSIONS)]
            images = [f for f in all_files if f.endswith(IMAGE_EXTENSIONS)]

            print(f"\n[{label_name.upper()}] {folder} → {len(videos)} videos, {len(images)} images")

            for path in tqdm(videos, desc="Extracting frames"):
                self._extract_frames(path, output_dir)

            for path in tqdm(images, desc="Copying images"):
                shutil.copy2(path, output_dir)

        self._summary()

    def _crawl(self, folder_path):
        """Rekursif cari semua file video & gambar dalam folder."""
        found = []
        for entry in os.listdir(folder_path):
            full_path = os.path.join(folder_path, entry)
            if os.path.isdir(full_path):
                found += self._crawl(full_path)
            elif entry.endswith(VIDEO_EXTENSIONS + IMAGE_EXTENSIONS):
                found.append(full_path)
        return found

    def _extract_frames(self, video_path, output_dir):
        """Ambil frame dari video setiap N frame, simpan sebagai jpg."""
        cap = cv2.VideoCapture(video_path)
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % self.frame_interval == 0:
                out_path = os.path.join(output_dir, f"{video_name}_f{frame_count:05d}.jpg")
                cv2.imwrite(out_path, frame)
            frame_count += 1

        cap.release()

    def _summary(self):
        print("\n=== Summary ===")
        for split in ["live", "spoof"]:
            path = os.path.join(self.output_root, split)
            n = len(os.listdir(path)) if os.path.exists(path) else 0
            print(f"  {split}: {n} files")

In [ ]:
OUTPUT_ROOT = "/content/frames"

FOLDER_LABEL = {
    "Selfies": 0,
    "3D_paper_mask ": 1,
    "Cutout_attacks": 1,
    "Latex_mask": 1,
    "Replay_display_attacks": 1,
    "Replay_mobile_attacks": 1,
    "Silicone_mask": 1,
    "Wrapped_3D_paper_mask": 1,
}

In [ ]:
for folder, label in FOLDER_LABEL.items():

  folder_path = os.path.join(DATASET_PATH, folder)
  label_name = "live" if label == 0 else "spoof"
  output_dir = os.path.join(OUTPUT_ROOT, label_name)
  os.makedirs(output_dir, exist_ok=True)

  videos = [f for f in os.listdir(folder_path) if f.endswith(VIDEO_EXTENSIONS)]

  for video in tqdm(videos):
    video_path = os.path.join(folder_path, video)
    extract_dataset(video_path, output_dir)

for split in ['live', 'spoof']:
  n = len(os.listdir(os.path.join(OUTPUT_ROOT, split)))